In [4]:
"""
B5. Ms sparse block-encoding validation experiments
---------------------------------------------------
This script tests and summarizes the matrix-selector sparse block-encoding
construction against a direct reference matrix.

It generates several structured and random selector vectors s, including
identity, cyclic shift, bit-reversal, constant, clustered, heavy-row,
and random patterns. For each case, it constructs the corresponding
reference matrix, builds the sparse block-encoding gate, extracts the
top-left encoded system block from the full unitary, and compares that
block to the expected normalized target matrix.

For each experiment, it computes:
    - problem size N and qubit count n,
    - row sparsity d_r,
    - normalization factor alpha = sqrt(d_r),
    - maximum entrywise absolute error,
    - Frobenius-norm error,
    - spectral-norm error,
    - pass/fail status based on allclose and tolerance checks.

When requested, it prints a formatted summary table of results across the
entire test suite.
"""

import _init_path
import numpy as np
from qiskit.quantum_info import Operator

from quantum.matrix_ms import (
    row_sparsity_from_s,
    get_ms_sparse_block_encoding_gate,
)


def reference_ms_matrix(s):
    """Construct the reference matrix-selector matrix from an index map.

    This routine converts an integer selector vector s into the corresponding
    matrix M whose j-th column is the standard basis vector e_{s[j]}.

    Args:
        s: One-dimensional array-like object of integer row indices.

    Returns:
        A complex NumPy array of shape (N, N), where N = len(s), containing
        the reference matrix-selector matrix.

    Raises:
        None.
    """
    s = np.asarray(s, dtype=int)
    N = len(s)
    M = np.zeros((N, N), dtype=complex)
    for j, i in enumerate(s):
        M[i, j] = 1.0
    return M


def extract_top_left_block_from_unitary(U, anc_qubits, sys_qubits):
    """Extract the encoded top-left system block from a block unitary.

    This routine interprets U as a unitary acting on ancilla and system
    registers, and extracts the block corresponding to ancilla state |0...0>
    on both input and output.

    Args:
        U: Full unitary matrix as a two-dimensional NumPy array.
        anc_qubits: Number of ancilla qubits.
        sys_qubits: Number of system qubits.

    Returns:
        A complex NumPy array of shape (2**sys_qubits, 2**sys_qubits)
        containing the top-left system block.

    Raises:
        None.
    """
    anc_dim = 2 ** anc_qubits
    sys_dim = 2 ** sys_qubits
    B = np.zeros((sys_dim, sys_dim), dtype=complex)
    for sys_out in range(sys_dim):
        for sys_in in range(sys_dim):
            row = anc_dim * sys_out
            col = anc_dim * sys_in
            B[sys_out, sys_in] = U[row, col]
    return B


def block_encoding_error_metrics(s):
    """Compute block-encoding accuracy metrics for a selector vector.

    This routine constructs the sparse block-encoding unitary associated
    with s, extracts its encoded system block, compares it to the normalized
    reference matrix, and returns a collection of error metrics.

    Args:
        s: One-dimensional array-like object of integer row indices.

    Returns:
        A dictionary containing problem parameters, error norms, pass/fail
        comparison information, and the compared matrices.

    Raises:
        None.
    """
    s = np.asarray(s, dtype=int)
    N = len(s)
    n = int(np.log2(N))
    d_r = row_sparsity_from_s(s)
    alpha = np.sqrt(d_r)

    gate = get_ms_sparse_block_encoding_gate(s)
    U = Operator(gate).data
    B = extract_top_left_block_from_unitary(U, anc_qubits=n + 1, sys_qubits=n)

    target = reference_ms_matrix(s) / alpha
    err = B - target

    return {
        "N": N,
        "n": n,
        "d_r": d_r,
        "alpha": alpha,
        "max_err": np.max(np.abs(err)),
        "fro_err": np.linalg.norm(err, ord="fro"),
        "spec_err": np.linalg.norm(err, ord=2),
        "allclose": np.allclose(B, target, atol=1e-10, rtol=1e-10),
        "B": B,
        "target": target,
    }


def check_ms_block_encoding_same_notebook(s, atol=1e-10, rtol=1e-10, verbose=True):
    """Check a single block-encoding instance and optionally print details.

    This routine constructs the block encoding for s, compares the extracted
    encoded block to the normalized reference matrix, and returns whether the
    comparison passes within the specified tolerances. When verbose is True,
    it prints matrices and error metrics.

    Args:
        s: One-dimensional array-like object of integer row indices.
        atol: Absolute tolerance used in the numerical comparison.
        rtol: Relative tolerance used in the numerical comparison.
        verbose: Whether to print detailed diagnostic information.

    Returns:
        True if the encoded block matches the target within tolerance, and
        False otherwise.

    Raises:
        None.
    """
    s = np.asarray(s, dtype=int)
    N = len(s)
    n = int(np.log2(N))
    d_r = row_sparsity_from_s(s)
    alpha = np.sqrt(d_r)

    gate = get_ms_sparse_block_encoding_gate(s)
    U = Operator(gate).data
    B = extract_top_left_block_from_unitary(U, anc_qubits=n + 1, sys_qubits=n)

    target = reference_ms_matrix(s) / alpha
    err = B - target

    passed = np.allclose(B, target, atol=atol, rtol=rtol)

    if verbose:
        print("s =", list(map(int, s)))
        print(f"N = {N}, n = {n}, d_r = {d_r}, alpha = {alpha}")
        print("B =")
        print(B)
        print("target =")
        print(target)
        print("pass =", passed)
        print("max err =", np.max(np.abs(err)))
        print("fro err =", np.linalg.norm(err, ord="fro"))
        print("spec err =", np.linalg.norm(err, ord=2))
        print()

    return passed


# ------------------------------------------------------------------
# Structured test-vector generators
# ------------------------------------------------------------------


def s_identity(N):
    """Generate the identity selector vector.

    This routine returns the selector vector s[j] = j, corresponding to the
    identity matrix-selector mapping.

    Args:
        N: Problem size.

    Returns:
        A one-dimensional NumPy array of length N containing the identity
        selector.

    Raises:
        None.
    """
    return np.arange(N, dtype=int)


def s_cyclic_shift(N):
    """Generate a cyclic-shift selector vector.

    This routine returns the selector vector s[j] = (j + 1) mod N,
    corresponding to a one-step cyclic shift.

    Args:
        N: Problem size.

    Returns:
        A one-dimensional NumPy array of length N containing the cyclic-shift
        selector.

    Raises:
        None.
    """
    return (np.arange(N, dtype=int) + 1) % N


def s_bit_reversal(N):
    """Generate a bit-reversal selector vector.

    This routine maps each index j to the integer obtained by reversing
    the binary representation of j using log2(N) bits.

    Args:
        N: Problem size, assumed to be a power of two.

    Returns:
        A one-dimensional NumPy array of length N containing the bit-reversal
        selector.

    Raises:
        None.
    """
    n = int(np.log2(N))
    out = np.zeros(N, dtype=int)
    for j in range(N):
        bits = format(j, f"0{n}b")
        out[j] = int(bits[::-1], 2)
    return out


def s_constant(N, value=0):
    """Generate a constant selector vector.

    This routine returns a selector vector whose entries are all equal to
    the specified row index value.

    Args:
        N: Problem size.
        value: Constant row index assigned to every column.

    Returns:
        A one-dimensional NumPy array of length N whose entries are all equal
        to value.

    Raises:
        None.
    """
    return np.full(N, int(value), dtype=int)


def s_two_cluster(N):
    """Generate a two-cluster selector vector.

    This routine creates a selector vector in which the first half of the
    columns map to row 0 and the second half map to row N // 2.

    Args:
        N: Problem size.

    Returns:
        A one-dimensional NumPy array of length N containing the two-cluster
        selector.

    Raises:
        None.
    """
    s = np.zeros(N, dtype=int)
    s[N // 2:] = N // 2
    return s


def s_heavy_row(N, heavy_mult=None):
    """Generate a selector vector with one heavily repeated row.

    This routine assigns many initial columns to row 0 and then maps the
    remaining columns injectively across other rows as much as possible.

    Args:
        N: Problem size.
        heavy_mult: Number of columns assigned to the heavy row. If None,
            the value N // 2 + 1 is used.

    Returns:
        A one-dimensional NumPy array of length N containing the heavy-row
        selector.

    Raises:
        None.
    """
    if heavy_mult is None:
        heavy_mult = N // 2 + 1

    heavy_mult = min(max(1, heavy_mult), N)
    s = np.empty(N, dtype=int)
    s[:heavy_mult] = 0

    next_rows = [r for r in range(1, N)]
    for j in range(heavy_mult, N):
        s[j] = next_rows[(j - heavy_mult) % len(next_rows)] if next_rows else 0

    return s


def s_random(N, rng):
    """Generate a random selector vector.

    This routine samples each entry of s independently and uniformly from
    the integers {0, 1, ..., N - 1}.

    Args:
        N: Problem size.
        rng: NumPy random number generator.

    Returns:
        A one-dimensional NumPy array of length N containing random row
        indices.

    Raises:
        None.
    """
    return rng.integers(low=0, high=N, size=N, dtype=int)


# ------------------------------------------------------------------
# ASCII / box-table formatting
# ------------------------------------------------------------------


def _fmt_int(x):
    """Format a value as an integer string.

    This routine converts the input value to an integer and returns its
    string representation.

    Args:
        x: Value to format.

    Returns:
        A string representation of x as an integer.

    Raises:
        None.
    """
    return str(int(x))


def _fmt_float(x, decimals=3):
    """Format a value in scientific notation.

    This routine converts the input value to float and returns a string
    formatted in scientific notation with the requested number of decimals.

    Args:
        x: Value to format.
        decimals: Number of digits after the decimal point.

    Returns:
        A string containing the scientific-notation representation of x.

    Raises:
        None.
    """
    return f"{float(x):.{decimals}e}"


def print_ms_results_table(rows):
    """Print experiment results in an ASCII box table.

    This routine formats a sequence of result dictionaries into a fixed-width
    Unicode table and prints the table to standard output.

    Args:
        rows: Iterable of dictionaries containing experiment summary fields.

    Returns:
        None.

    Raises:
        None.
    """
    headers = ["family", "N", "n", "d_r", "alpha", "max_err", "fro_err", "spec_err", "pass"]
    aligns = ["L", "R", "R", "R", "R", "R", "R", "R", "R"]

    table_rows = []
    for row in rows:
        table_rows.append([
            str(row["family"]),
            _fmt_int(row["N"]),
            _fmt_int(row["n"]),
            _fmt_int(row["d_r"]),
            f"{float(row['alpha']):.6f}",
            _fmt_float(row["max_err"], 3),
            _fmt_float(row["fro_err"], 3),
            _fmt_float(row["spec_err"], 3),
            str(bool(row["pass"])),
        ])

    widths = [len(h) for h in headers]
    for row in table_rows:
        for j, cell in enumerate(row):
            widths[j] = max(widths[j], len(cell))

    TL, TR, BL, BR = "┌", "┐", "└", "┘"
    H, V = "─", "│"
    TJ, BJ, LJ, RJ, CJ = "┬", "┴", "├", "┤", "┼"

    def hline(left, mid, right):
        """Construct a horizontal table border.

        This routine builds a horizontal rule using the current column widths
        and the specified left, middle, and right border characters.

        Args:
            left: Left border character.
            mid: Column-separator character.
            right: Right border character.

        Returns:
            A string containing the formatted horizontal rule.

        Raises:
            None.
        """
        return left + mid.join(H * (w + 2) for w in widths) + right

    def format_cell(text, w, align, header=False):
        """Format a single table cell with alignment and padding.

        This routine centers header cells and left- or right-aligns body cells
        according to the requested alignment code.

        Args:
            text: Cell content.
            w: Cell width.
            align: Alignment specifier, typically "L" or "R".
            header: Whether the cell belongs to the header row.

        Returns:
            A padded string representing the formatted cell.

        Raises:
            None.
        """
        if header:
            return f" {text:^{w}} "
        if align == "R":
            return f" {text:>{w}} "
        return f" {text:<{w}} "

    def render_row(row, header=False):
        """Render a complete table row.

        This routine formats each cell in the given row and joins them with
        vertical separators to produce one printable table row.

        Args:
            row: Sequence of cell strings.
            header: Whether the row is the header row.

        Returns:
            A string containing the rendered table row.

        Raises:
            None.
        """
        return V + V.join(
            format_cell(row[j], widths[j], aligns[j], header=header)
            for j in range(len(headers))
        ) + V

    print(hline(TL, TJ, TR))
    print(render_row(headers, header=True))
    print(hline(LJ, CJ, RJ))
    for row in table_rows:
        print(render_row(row))
    print(hline(BL, BJ, BR))


# ------------------------------------------------------------------
# Batch experiment driver
# ------------------------------------------------------------------


def generate_test_suite(n_values=(1, 2, 3, 4, 5), random_trials=5, seed=12345):
    """Generate the full collection of test selector vectors.

    This routine builds a suite of structured and random selector vectors
    across the requested qubit sizes.

    Args:
        n_values: Iterable of qubit counts, with N = 2**n for each entry.
        random_trials: Number of random selector instances to generate per N.
        seed: Seed used to initialize the random number generator.

    Returns:
        A list of pairs (name, s), where name identifies the selector family
        and s is the corresponding NumPy array.

    Raises:
        None.
    """
    rng = np.random.default_rng(seed)
    suite = []

    for n in n_values:
        N = 2 ** n

        suite.append(("identity", s_identity(N)))
        suite.append(("cyclic_shift", s_cyclic_shift(N)))
        suite.append(("bit_reversal", s_bit_reversal(N)))
        suite.append(("constant_0", s_constant(N, 0)))
        suite.append(("constant_last", s_constant(N, N - 1)))
        suite.append(("two_cluster", s_two_cluster(N)))
        suite.append(("heavy_row", s_heavy_row(N)))

        for k in range(random_trials):
            suite.append((f"random_{k}", s_random(N, rng)))

    return suite


def run_ms_block_encoding_experiments(
    n_values=(1, 2, 3, 4, 5),
    random_trials=5,
    seed=12345,
    atol=1e-10,
    rtol=1e-10,
    print_table=True,
):
    """Run the block-encoding experiment suite and summarize outcomes.

    This routine generates the test suite, evaluates block-encoding error
    metrics for each selector vector, records pass/fail information, and
    optionally prints a formatted results table.

    Args:
        n_values: Iterable of qubit counts, with N = 2**n for each entry.
        random_trials: Number of random selector instances to generate per N.
        seed: Seed used to initialize the random number generator.
        atol: Absolute tolerance used in pass/fail checks.
        rtol: Relative tolerance used in pass/fail checks.
        print_table: Whether to print the summary table.

    Returns:
        A pair (rows, failures), where rows is a list of summary dictionaries
        for all experiments and failures contains the failed test cases.

    Raises:
        None.
    """
    suite = generate_test_suite(
        n_values=n_values,
        random_trials=random_trials,
        seed=seed,
    )

    rows = []
    failures = []

    for name, s in suite:
        metrics = block_encoding_error_metrics(s)
        passed = (
            metrics["max_err"] <= atol + rtol * np.max(np.abs(metrics["target"]))
            and metrics["allclose"]
        )

        row = {
            "family": name,
            "N": metrics["N"],
            "n": metrics["n"],
            "d_r": metrics["d_r"],
            "alpha": metrics["alpha"],
            "max_err": metrics["max_err"],
            "fro_err": metrics["fro_err"],
            "spec_err": metrics["spec_err"],
            "pass": passed,
        }
        rows.append(row)

        if not passed:
            failures.append((name, s, row))

    if print_table:
        print_ms_results_table(rows)

    return rows, failures

#check_ms_block_encoding_same_notebook([0, 1, 2, 3])
#check_ms_block_encoding_same_notebook([0, 0, 0, 0])
#check_ms_block_encoding_same_notebook([0, 0, 2, 2])

In [2]:
rows, failures = run_ms_block_encoding_experiments(
    n_values=(1, 2, 3, 4, 5),
    random_trials=10,
    seed=7,
)
print("number of failures =", len(failures))

┌───────────────┬────┬───┬─────┬──────────┬───────────┬───────────┬───────────┬──────┐
│    family     │ N  │ n │ d_r │  alpha   │  max_err  │  fro_err  │ spec_err  │ pass │
├───────────────┼────┼───┼─────┼──────────┼───────────┼───────────┼───────────┼──────┤
│ identity      │  2 │ 1 │   1 │ 1.000000 │ 0.000e+00 │ 0.000e+00 │ 0.000e+00 │ True │
│ cyclic_shift  │  2 │ 1 │   1 │ 1.000000 │ 0.000e+00 │ 0.000e+00 │ 0.000e+00 │ True │
│ bit_reversal  │  2 │ 1 │   1 │ 1.000000 │ 0.000e+00 │ 0.000e+00 │ 0.000e+00 │ True │
│ constant_0    │  2 │ 1 │   2 │ 1.414214 │ 2.220e-16 │ 2.483e-16 │ 2.483e-16 │ True │
│ constant_last │  2 │ 1 │   2 │ 1.414214 │ 2.220e-16 │ 2.483e-16 │ 2.483e-16 │ True │
│ two_cluster   │  2 │ 1 │   1 │ 1.000000 │ 0.000e+00 │ 0.000e+00 │ 0.000e+00 │ True │
│ heavy_row     │  2 │ 1 │   2 │ 1.414214 │ 2.220e-16 │ 2.483e-16 │ 2.483e-16 │ True │
│ random_0      │  2 │ 1 │   2 │ 1.414214 │ 2.220e-16 │ 2.483e-16 │ 2.483e-16 │ True │
│ random_1      │  2 │ 1 │   2 │ 1.414214 │